Weapons and Ammunition data analysis


In [1]:
import pandas as pd
pd.options.display.max_columns = None

df = pd.read_csv('data/amcdata_weapons_facilities_V2.csv', encoding='latin-1')
df = df[df['summary_category'] != 1]
facilities_df = df[df['item_type'] == 1]

### Select weapons life cycle stages

In [2]:
cols_to_keep = [
    # Identifiers
    'item_type', 'item',

    # Lifecycle stages - ban flags
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',

    # Lifecycle stages - restriction flags
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal',

    # End-of-life stages (no ban/restriction equivalents)
    'eliminitation',             # note: typo in the dataset
    'conversion',
    'modernization',
    'facility_destruction',
]

facilities_lifecycle_df = facilities_df[cols_to_keep]


Which columns are causing problems due to no variance?

In [6]:
ban_cols = [c for c in facilities_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in facilities_lifecycle_df.columns if 'restriction' in c]

flag_cols = ban_cols + restriction_cols

for col in flag_cols:
    unique_vals = facilities_lifecycle_df[col].dropna().unique()
    if len(unique_vals) <= 1:
        print(f"{col}: only contains {unique_vals}")

ban_possession: only contains [0]
ban_disposal: only contains [0]


### Testing correlations between different life cycle stages of weapons/ammunitions

In [9]:
# Separate ban and restriction columns
ban_cols = [c for c in facilities_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in facilities_lifecycle_df.columns if 'restriction' in c]

# Correlation between every ban col vs every restriction col
corr_matrix = facilities_lifecycle_df[ban_cols + restriction_cols].corr()

# Slice to only show ban vs restriction (not ban vs ban or restriction vs restriction)
corr_ban_vs_restriction = corr_matrix.loc[ban_cols, restriction_cols]
print(corr_ban_vs_restriction)

                 restriction_development  testing_restriction  \
ban_development                -0.116138            -0.111763   
ban_testing                    -0.148427            -0.142836   
ban_production                 -0.151433            -0.106221   
ban_acquisition                -0.089313            -0.085949   
ban_possession                       NaN                  NaN   
ban_station                    -0.080417            -0.077388   
ban_transfer                    0.012698             0.021226   
ban_use                        -0.112598            -0.108357   
ban_disposal                         NaN                  NaN   

                 restriction_production  restriction_acquisition  \
ban_development               -0.124676                -0.102742   
ban_testing                   -0.159338                -0.131306   
ban_production                -0.126048                -0.133966   
ban_acquisition               -0.095879                -0.079011   
ban_posse

In [11]:
# Stack matrix into a series and sort
corr_ranked = (
    corr_ban_vs_restriction
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'ban', 'level_1': 'restriction', 0: 'correlation'})
    .sort_values('correlation', ascending=False)
)

print(corr_ranked.head(40))

                ban              restriction  correlation
44     ban_transfer   restriction_possession     0.046764
41     ban_transfer      testing_restriction     0.021226
40     ban_transfer  restriction_development     0.012698
39      ban_station     restriction_disposal    -0.013455
31  ban_acquisition     restriction_disposal    -0.014944
55          ban_use     restriction_disposal    -0.018840
7   ban_development     restriction_disposal    -0.019432
47     ban_transfer     restriction_disposal    -0.020581
15      ban_testing     restriction_disposal    -0.024834
23   ban_production     restriction_disposal    -0.025337
38      ban_station          restriction_use    -0.034286
52          ban_use   restriction_possession    -0.041622
30  ban_acquisition          restriction_use    -0.050038
14      ban_testing          restriction_use    -0.067187
37      ban_station     restriction_transfer    -0.067903
35      ban_station  restriction_acquisition    -0.071141
29  ban_acquis

In [13]:
# Pearson - default, fine for binary
facilities_lifecycle_df[ban_cols + restriction_cols].corr(method='pearson')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,1.000000,0.485126,0.522847,0.389506,NaN,0.191241,0.311692,0.226493,NaN,-0.116138,-0.111763,-0.124676,-0.102742,-0.219030,-0.098065,-0.135045,-0.019432
ban_testing,0.485126,1.000000,0.659295,0.414647,NaN,0.335913,0.025691,0.097281,NaN,-0.148427,-0.142836,-0.159338,-0.131306,-0.294941,-0.125329,-0.067187,-0.024834
ban_production,0.522847,0.659295,1.000000,0.589785,NaN,0.395833,0.207358,0.242429,NaN,-0.151433,-0.106221,-0.126048,-0.133966,-0.301866,-0.127867,-0.141476,-0.025337
ban_acquisition,0.389506,0.414647,0.589785,1.000000,NaN,0.585049,0.291954,0.325687,NaN,-0.089313,-0.085949,-0.095879,-0.079011,-0.193739,-0.075414,-0.050038,-0.014944
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,0.191241,0.335913,0.395833,0.585049,NaN,1.000000,0.016763,0.113955,NaN,-0.080417,-0.077388,-0.086329,-0.071141,-0.174441,-0.067903,-0.034286,-0.013455
ban_transfer,0.311692,0.025691,0.207358,0.291954,NaN,0.016763,1.000000,0.207099,NaN,0.012698,0.021226,-0.132050,-0.108819,0.046764,-0.103865,-0.102268,-0.020581
ban_use,0.226493,0.097281,0.242429,0.325687,NaN,0.113955,0.207099,1.000000,NaN,-0.112598,-0.108357,-0.120876,-0.099611,-0.041622,-0.095076,-0.130929,-0.018840
ban_disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restriction_development,-0.116138,-0.148427,-0.151433,-0.089313,NaN,-0.080417,0.012698,-0.112598,NaN,1.000000,0.194218,0.044062,-0.098255,-0.056645,-0.047894,-0.048401,-0.026329


In [14]:
# Spearman - better for ordinal/binary data, more robust
facilities_lifecycle_df[ban_cols + restriction_cols].corr(method='spearman')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,1.000000,0.485126,0.522847,0.389506,NaN,0.191241,0.311692,0.226493,NaN,-0.116138,-0.111763,-0.124676,-0.102742,-0.219030,-0.098065,-0.135045,-0.019432
ban_testing,0.485126,1.000000,0.659295,0.414647,NaN,0.335913,0.025691,0.097281,NaN,-0.148427,-0.142836,-0.159338,-0.131306,-0.294941,-0.125329,-0.067187,-0.024834
ban_production,0.522847,0.659295,1.000000,0.589785,NaN,0.395833,0.207358,0.242429,NaN,-0.151433,-0.106221,-0.126048,-0.133966,-0.301866,-0.127867,-0.141476,-0.025337
ban_acquisition,0.389506,0.414647,0.589785,1.000000,NaN,0.585049,0.291954,0.325687,NaN,-0.089313,-0.085949,-0.095879,-0.079011,-0.193739,-0.075414,-0.050038,-0.014944
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,0.191241,0.335913,0.395833,0.585049,NaN,1.000000,0.016763,0.113955,NaN,-0.080417,-0.077388,-0.086329,-0.071141,-0.174441,-0.067903,-0.034286,-0.013455
ban_transfer,0.311692,0.025691,0.207358,0.291954,NaN,0.016763,1.000000,0.207099,NaN,0.012698,0.021226,-0.132050,-0.108819,0.046764,-0.103865,-0.102268,-0.020581
ban_use,0.226493,0.097281,0.242429,0.325687,NaN,0.113955,0.207099,1.000000,NaN,-0.112598,-0.108357,-0.120876,-0.099611,-0.041622,-0.095076,-0.130929,-0.018840
ban_disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restriction_development,-0.116138,-0.148427,-0.151433,-0.089313,NaN,-0.080417,0.012698,-0.112598,NaN,1.000000,0.194218,0.044062,-0.098255,-0.056645,-0.047894,-0.048401,-0.026329
